# Pendulum swing-up: dynamic programming vs reinforcement learning (PPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/courses/gro860/labs/pendulum_dp_vs_ppo.ipynb)

This page compares two approaches for generating a global swing-up controller for a torque-limited pendulum, based on the same cost function:

1. **Dynamic programming (value iteration)**: a numerical solution of the Bellman equation over a discretized state space, leading to the global optimal policy.
2. **Reinforcement learning (PPO)**: a neural-network policy trained by trial and error on a gym environment whose reward is the negative of the same cost.

The pendulum here uses the "inverted" convention: $\theta = 0$ is the upright target and the swing-up starts at the bottom, $\theta = -\pi$. The torque limit is low enough that a direct swing-up is impossible: both methods must discover the "pumping" strategy.

We use the toolbox [minilink](https://github.com/alx87grd/minilink) for the dynamics, DP solver, simulation and animation, and [stable-baselines3](https://stable-baselines3.readthedocs.io) for the RL algorithm.

In [ ]:
# Local: minilink already installed. Colab: clone + path + RL dependencies.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.costs import QuadraticCost
from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import InvertedPendulum
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym
from minilink.planning.policy_synthesis import plotting
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem

# Dynamics

An inverted pendulum with physical parameters similar to the classic `Pendulum-v1` gym benchmark: the maximum torque (2 Nm) is well below what a direct swing-up would require (5 Nm of gravity torque at the horizontal).

In [ ]:
def make_pendulum():
    plant = InvertedPendulum()

    # Physical parameters (similar to the gym Pendulum-v1 benchmark)
    plant.params["gravity"] = 10.0
    plant.params["m"] = 1.0
    plant.params["l"] = 0.5  # distance to the center of mass
    plant.params["I"] = 1.0 / 12.0  # inertia about the center of mass

    # Min/max state and control inputs
    plant.state.lower_bound = np.array([-2.0 * np.pi, -8.0])
    plant.state.upper_bound = np.array([+2.0 * np.pi, +8.0])
    plant.inputs["u"].lower_bound = np.array([-2.0])
    plant.inputs["u"].upper_bound = np.array([+2.0])

    plant.x0 = np.array([-np.pi, 0.0])  # start at the bottom
    return plant


plant = make_pendulum()

# Cost function

Both methods use the same quadratic cost $J = \int (x'Qx + u'Ru) \, dt$ about the upright target $\bar x = [0, 0]$. The weights are scaled to mimic the reward of the classic gym pendulum benchmark, $r = -(\theta^2 + 0.1\,\dot\theta^2 + 0.001\,\tau^2)$ per time step of $\Delta t = 0.05$ s.

In [ ]:
dt = 0.05
X_TARGET = np.array([0.0, 0.0])  # upright

cost = QuadraticCost.from_system(
    plant,
    xbar=X_TARGET,
    Q=np.diag([1.0 / dt, 0.1 / dt]),
    R=np.diag([0.001 / dt]),
)

# Solution 1: dynamic programming (value iteration)

Discretize the state and input domains, then solve the Bellman equation by value iteration.

In [ ]:
problem = PlanningProblem(plant, x_goal=X_TARGET, cost=cost)

grid = StateSpaceGrid(problem, x_grid_shape=(201, 201), u_grid_shape=(21,), dt=dt)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0, tol=1.0, max_iterations=2000, out_of_bound_cost=10000.0, verbose=True
    ),
)

result = planner.solve().policy
planner.clean_infeasible_set()

dp_ctl = result.controller()

In [ ]:
plotting.plot_value(grid, result.J, vmax=1000.0, title="DP cost-to-go")
plotting.plot_policy(grid, result.pi)

# Solution 2: reinforcement learning (PPO)

We expose the same system and cost as a gym environment: the reward is $r = -g(x,u,t)\,\Delta t$, so maximizing the cumulative reward is equivalent to minimizing the cost $J$. The initial states of the training episodes are distributed uniformly over the domain around the bottom position.

In [ ]:
env = Sys2Gym(plant, cost, dt=dt, tf=10.0)

env.clipping_states = True  # reproduce the behaviour of the gym pendulum

env.reset_mode = "uniform"
env.x0_lb = np.array([-np.pi, -1.0])
env.x0_ub = np.array([+np.pi, +1.0])

In [ ]:
from stable_baselines3 import PPO

nn = PPO("MlpPolicy", env, verbose=0)
nn.learn(total_timesteps=250000)
print("PPO training done")

# Comparing the policies

The learned PPO policy is evaluated over the same state grid as the DP solution. The closer the training gets to convergence, the more the learned policy should resemble the globally optimal DP policy (in the regions of the state space visited during training).

In [ ]:
ppo_ctl = SB3Controller(nn)

# PPO policy evaluated on the DP grid nodes
u_ppo_map, _ = nn.predict(grid.states.astype(np.float32), deterministic=True)

plotting.plot_policy(grid, result.pi)  # DP policy
plotting.plot_value(
    grid, u_ppo_map[:, 0], vmin=-2.0, vmax=2.0, cmap="bwr", title="PPO policy u[0]"
)

# Closed-loop simulations

Both controllers are wired to the pendulum and simulated from the bottom position $[\theta = -\pi, \dot\theta = 0]$.

In [ ]:
x0 = np.array([-np.pi, 0.0])
tf = 10.0


def closed_loop(controller, x0, name):
    plant = make_pendulum()
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    traj = diagram.compute_trajectory(tf=tf, n_steps=2001, solver="euler")
    return diagram, traj


cl_dp, traj_dp = closed_loop(dp_ctl, x0, "Pendulum with DP controller")
cl_ppo, traj_ppo = closed_loop(ppo_ctl, x0, "Pendulum with PPO controller")

cl_dp.plot_trajectory(traj_dp)
cl_ppo.plot_trajectory(traj_ppo)

Animation with the DP controller:

In [ ]:
cl_dp.animate(traj_dp)

Animation with the PPO controller:

In [ ]:
cl_ppo.animate(traj_ppo)

# Performance

Comparison of the two solutions in terms of the defined cost function $J = \int (x'Qx + u'Ru) \, dt$. The DP solution is the global optimum for the discretized problem; a well-trained PPO policy should approach (but not beat) it.

In [ ]:
# Rebuild the applied torques from each policy, then evaluate the same cost
u_dp = np.array([dp_ctl.action(x) for x in traj_dp.x.T]).T
u_ppo, _ = nn.predict(traj_ppo.x.T.astype(np.float32), deterministic=True)

traj_dp_cost = cost.evaluate_trajectory(Trajectory(t=traj_dp.t, x=traj_dp.x, u=u_dp))
traj_ppo_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_ppo.t, x=traj_ppo.x, u=u_ppo.T)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_dp_cost.t, traj_dp_cost.signals["cost"][0], label="DP")
ax.plot(traj_ppo_cost.t, traj_ppo_cost.signals["cost"][0], label="PPO")
ax.set_xlabel("t [s]")
ax.set_ylabel("$J = \\int (x'Qx + u'Ru) \\, dt$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("DP  | total cost:", round(float(traj_dp_cost.signals["cost"][0, -1]), 1))
print("PPO | total cost:", round(float(traj_ppo_cost.signals["cost"][0, -1]), 1))